# Oil Spill Viewer — Server Launcher

Run the cells below in order. After Cell 3, open the URL printed to access the viewer.

In [ ]:
# Cell 1 — Install jupyter-server-proxy if not already installed
import subprocess, sys
result = subprocess.run([sys.executable, '-m', 'pip', 'install', 
                         'jupyter-server-proxy', 'fastapi', 'uvicorn[standard]',
                         'boto3', 'rasterio', 'Pillow', 'numpy', '-q'],
                        capture_output=True, text=True)
print(result.stdout[-500:] if result.stdout else '(no output)')
print(result.stderr[-300:] if result.stderr else '')
print('Done — restart the Jupyter server if this is the first time installing jupyter-server-proxy')

In [ ]:
# Cell 2 — Check if jupyter-server-proxy is working
import os, subprocess

jhub_user   = os.environ.get('JUPYTERHUB_USER', os.environ.get('USER', 'unknown'))
jhub_prefix = os.environ.get('JUPYTERHUB_SERVICE_PREFIX', f'/user/{jhub_user}/')
port        = 8050

# Detect base URL
base_url = os.environ.get('JUPYTERHUB_BASE_URL', 'https://jupyter.glodal-inc.net')
proxy_url = f"{base_url.rstrip('/')}/user/{jhub_user}/proxy/{port}/"

print(f'JupyterHub user    : {jhub_user}')
print(f'JupyterHub prefix  : {jhub_prefix}')
print()
print('After starting the server (Cell 3), open:')
print(f'  {proxy_url}')
print()
print('If you see 404 nginx, restart the Jupyter server after installing jupyter-server-proxy')

In [ ]:
# Cell 3 — Start the Oil Spill Viewer backend
# This cell runs the server. Keep the notebook open while using the viewer.
# Interrupt the kernel (■ button) to stop the server.

import os, sys
from pathlib import Path

# ── Find the backend ────────────────────────────────────────────────────────
# Try to find app.py relative to this notebook, then fall back to common paths
notebook_dir = Path(os.getcwd())
candidates = [
    notebook_dir / 'backend' / 'app.py',
    notebook_dir / 'oilspill_viewer' / 'backend' / 'app.py',
    notebook_dir.parent / 'backend' / 'app.py',
]
app_path = next((p for p in candidates if p.exists()), None)

if app_path is None:
    # Manual fallback — edit this path if needed
    app_path = Path('/home/jovyan/shared/Sachin_Rayamajhi/OIL SPILL SEGMENTATION_1BAND/Pre_step_before_web/oilspill_viewer/oilspill_viewer/backend/app.py')

print(f'Backend: {app_path}')
assert app_path.exists(), f'app.py not found at {app_path}. Edit the path above.'

backend_dir = app_path.parent

# ── Set credentials + config ─────────────────────────────────────────────────
os.environ.update({
    'AWS_ACCESS_KEY_ID':                   'PB1VCH7O58UFUM53PTBT',
    'AWS_SECRET_ACCESS_KEY':               'vK3ZpOC94kcCj94TWTnwg5FvMk288BLCCKlvCfnj',
    'AWS_DEFAULT_REGION':                  'us-east-1',
    'AWS_REQUEST_CHECKSUM_CALCULATION':    'when_required',
    'S3_ENDPOINT':                         'https://rgw.glodal-inc.net',
    'S3_BUCKET':                           'Inference_Oil_Spill_segmentation',
    'S3_MASK_ROOT':                        'oil_spill_brazil/Output/Oil_Spill_Postprocessed_v15',
    'S3_SAR_ROOT':                         'oil_spill_brazil/Output/Preprocess_After_SNAP_',
    'S3_CACHE_ROOT':                       'oil_spill_brazil/Output/Viewer_Cache',
    'USE_S3_CACHE':                        'true',
    'MAX_SAR_DIM':                         '3000',
    'PORT':                                '8050',
})

# ── Show access URL ──────────────────────────────────────────────────────────
jhub_user = os.environ.get('JUPYTERHUB_USER', os.environ.get('USER', 'sachin11-2dgeomatics'))
print()
print('=' * 60)
print('  Open in browser:')
print(f'  https://jupyter.glodal-inc.net/user/{jhub_user}/proxy/8050/')
print('=' * 60)
print()
print('Starting server... (interrupt kernel to stop)')
print()

# ── Run the server ───────────────────────────────────────────────────────────
os.chdir(backend_dir)
sys.path.insert(0, str(backend_dir))

import uvicorn

# Import the app from app.py
import importlib.util
spec = importlib.util.spec_from_file_location('app', app_path)
mod  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

root_path = os.environ.get('JUPYTERHUB_SERVICE_PREFIX', f'/user/{jhub_user}/')

uvicorn.run(
    mod.app,
    host='0.0.0.0',
    port=8050,
    log_level='info',
    root_path=root_path,
)

In [ ]:
# Cell 4 (optional) — Check pre-cache status
import boto3
s3 = boto3.client('s3',
    endpoint_url='https://rgw.glodal-inc.net',
    aws_access_key_id='PB1VCH7O58UFUM53PTBT',
    aws_secret_access_key='vK3ZpOC94kcCj94TWTnwg5FvMk288BLCCKlvCfnj',
    region_name='us-east-1')

cache_root = 'oil_spill_brazil/Output/Viewer_Cache'
pages = s3.get_paginator('list_objects_v2').paginate(
    Bucket='Inference_Oil_Spill_segmentation', Prefix=cache_root)

meta_keys = []
for pg in pages:
    for obj in pg.get('Contents', []):
        if obj['Key'].endswith('meta.json'):
            meta_keys.append(obj['Key'])

print(f'Cached scenes: {len(meta_keys)} / 21')
for k in sorted(meta_keys):
    sid = k.split('/')[-2]
    print(f'  ✅ {sid}')